# VSR Dataset — Statistici

Analiză interactivă din `data/catalog/dataset.db`. Rulează celulele în ordine.

In [ ]:
import sqlite3
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams.update({'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#16213e',
                     'axes.edgecolor': '#444', 'grid.color': '#333',
                     'text.color': 'white', 'axes.labelcolor': 'white',
                     'xtick.color': 'white', 'ytick.color': 'white',
                     'figure.dpi': 130})

REPO = Path(os.getcwd()).parent  # assumes notebooks/ is one level inside repo root
DB = REPO / 'data' / 'catalog' / 'dataset.db'
assert DB.exists(), f'DB not found: {DB}'

conn = sqlite3.connect(DB)
conn.row_factory = sqlite3.Row
print(f'Connected: {DB}')

## 1. KPI-uri generale

In [ ]:
q = conn.execute("""
    SELECT
        COUNT(*) FILTER (WHERE review_status != 'rejected' OR review_status IS NULL) AS total_segments,
        ROUND(SUM(duration) FILTER (WHERE review_status != 'rejected' OR review_status IS NULL) / 3600.0, 2) AS total_hours,
        COUNT(*) FILTER (WHERE review_status = 'approved') AS approved,
        COUNT(*) FILTER (WHERE review_status = 'rejected') AS rejected,
        COUNT(*) FILTER (WHERE review_status IS NULL OR review_status = '') AS pending,
        COUNT(DISTINCT video_id) AS videos
    FROM segments
""").fetchone()

for k in q.keys():
    print(f'{k:25s}: {q[k]}')

## 2. Distribuție pe tier și regiune

In [ ]:
df = pd.read_sql("""
    SELECT s.quality_tier, v.region,
           COUNT(*) AS segments,
           ROUND(SUM(s.duration) / 3600.0, 3) AS hours
    FROM segments s
    LEFT JOIN videos v USING (video_id)
    WHERE COALESCE(s.review_status, '') != 'rejected'
    GROUP BY 1, 2
    ORDER BY 1, 2
""", conn)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

tier_df = df.groupby('quality_tier')[['segments', 'hours']].sum()
colors = {'A': '#34d399', 'B': '#22d3ee', 'C': '#f97316'}
tier_df['segments'].plot.bar(ax=axes[0], color=[colors.get(t, '#888') for t in tier_df.index])
axes[0].set_title('Segmente per Tier'); axes[0].set_xlabel(''); axes[0].tick_params(axis='x', rotation=0)

reg_df = df.groupby('region')[['hours']].sum().sort_values('hours', ascending=False)
reg_df['hours'].plot.bar(ax=axes[1], color='#22d3ee')
axes[1].set_title('Ore per Regiune'); axes[1].set_xlabel(''); axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()
print(df.to_string(index=False))

## 3. Distribuție durată segmente

In [ ]:
dur = pd.read_sql("""
    SELECT duration FROM segments
    WHERE COALESCE(review_status, '') != 'rejected'
    AND duration IS NOT NULL AND duration > 0
""", conn)['duration']

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(dur.clip(upper=20), bins=80, color='#34d399', alpha=0.85, edgecolor='none')
ax.set_xlabel('Durată (s)'); ax.set_ylabel('Segmente')
ax.set_title(f'Distribuție durată — {len(dur):,} segmente | median={dur.median():.1f}s | mean={dur.mean():.1f}s')
ax.axvline(dur.median(), color='#f97316', linestyle='--', linewidth=1.5, label=f'median {dur.median():.1f}s')
ax.legend()
plt.tight_layout()
plt.show()

## 4. WER: standard vs dialectat medium

In [ ]:
wer_df = pd.read_sql("""
    SELECT wer_standard_vs_dialectat_mediu AS wer, quality_tier
    FROM segments
    WHERE wer_standard_vs_dialectat_mediu IS NOT NULL
    AND COALESCE(review_status, '') != 'rejected'
""", conn).dropna()

print(f'Segmente cu WER calculat: {len(wer_df):,}')
print(wer_df['wer'].describe().round(3))
print(f"\nZona ideala (0.05-0.20): {((wer_df['wer'] >= 0.05) & (wer_df['wer'] <= 0.20)).sum():,} ({((wer_df['wer'] >= 0.05) & (wer_df['wer'] <= 0.20)).mean()*100:.1f}%)")
print(f"WER = 0 (fara schimb):    {(wer_df['wer'] == 0).sum():,} ({(wer_df['wer'] == 0).mean()*100:.1f}%)")
print(f"WER > 0.40 (grav):        {(wer_df['wer'] > 0.40).sum():,} ({(wer_df['wer'] > 0.40).mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(wer_df['wer'].clip(upper=1.0), bins=50, color='#f97316', alpha=0.85, edgecolor='none')
axes[0].axvline(0.05, color='#34d399', linestyle='--', linewidth=1, label='0.05')
axes[0].axvline(0.20, color='#f97316', linestyle='--', linewidth=1, label='0.20')
axes[0].set_title('WER standard vs dialectat mediu'); axes[0].set_xlabel('WER'); axes[0].legend()

wer_tier = wer_df.groupby('quality_tier')['wer'].median()
wer_tier.plot.bar(ax=axes[1], color=[colors.get(t, '#888') for t in wer_tier.index])
axes[1].set_title('WER median per Tier'); axes[1].set_xlabel(''); axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 5. Progres procesare Whisper dialectat

In [ ]:
q = conn.execute("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE whisper_standard_mediu IS NOT NULL AND whisper_standard_mediu != '') AS standard_mediu,
        COUNT(*) FILTER (WHERE whisper_dialectat_mediu IS NOT NULL AND whisper_dialectat_mediu != '') AS dialectat_mediu,
        COUNT(*) FILTER (WHERE whisper_dialectat_large IS NOT NULL AND whisper_dialectat_large != '') AS dialectat_large,
        COUNT(*) FILTER (WHERE transcript_source IS NOT NULL AND transcript_source != '') AS reviewed_source
    FROM segments
    WHERE COALESCE(review_status, '') != 'rejected'
""").fetchone()

total = q['total']
for k in ['standard_mediu', 'dialectat_mediu', 'dialectat_large', 'reviewed_source']:
    v = q[k]
    print(f'{k:25s}: {v:6,}  ({v/total*100:.1f}%)')

fig, ax = plt.subplots(figsize=(8, 3))
labels = ['Standard\nmediu', 'Dialectat\nmediu', 'Dialectat\nlarge', 'Reviewed\n(source sel.)']
vals = [q['standard_mediu'], q['dialectat_mediu'], q['dialectat_large'], q['reviewed_source']]
bars = ax.bar(labels, vals, color=['#22d3ee', '#34d399', '#f97316', '#a78bfa'])
ax.axhline(total, color='#888', linestyle='--', linewidth=1, label=f'Total {total:,}')
ax.set_title('Progres transcrieri Whisper'); ax.set_ylabel('Segmente')
ax.legend()
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{v/total*100:.0f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 6. Top canale / videouri

In [ ]:
channels = pd.read_sql("""
    SELECT
        COALESCE(v.channel_name, v.video_id) AS channel,
        COUNT(DISTINCT v.video_id) AS videos,
        COUNT(s.segment_id) AS segments,
        ROUND(SUM(s.duration) / 3600.0, 2) AS hours
    FROM videos v
    LEFT JOIN segments s USING (video_id)
    WHERE COALESCE(s.review_status, '') != 'rejected'
    GROUP BY 1
    ORDER BY hours DESC
    LIMIT 20
""", conn)

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(channels['channel'][::-1], channels['hours'][::-1], color='#22d3ee', alpha=0.85)
ax.set_xlabel('Ore extrase'); ax.set_title('Top 20 canale după ore extrase')
plt.tight_layout()
plt.show()

print(channels.to_string(index=False))

## 7. Demografie vorbitori

In [ ]:
spk = pd.read_sql("""
    SELECT sp.gender, sp.age_estimate,
           ROUND(SUM(s.duration) / 60.0, 1) AS minutes,
           COUNT(s.segment_id) AS segments
    FROM speakers sp
    JOIN segments s USING (speaker_id)
    WHERE COALESCE(s.review_status, '') != 'rejected'
    GROUP BY sp.speaker_id, sp.gender, sp.age_estimate
""", conn)

if len(spk) == 0:
    print('Nicio informatie despre vorbitori.')
else:
    print(f'Total vorbitori unici: {len(spk)}')
    print("\nDistributie gen:")
    print(spk.groupby('gender')[['segments', 'minutes']].sum().round(1))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    gen = spk.groupby('gender')['minutes'].sum()
    gen.plot.pie(ax=axes[0], autopct='%1.0f%%', colors=['#34d399', '#f97316', '#22d3ee'],
                 startangle=90, textprops={'fontsize': 11})
    axes[0].set_title('Minute per gen'); axes[0].set_ylabel('')

    spk['age_estimate'].dropna().plot.hist(ax=axes[1], bins=20, color='#a78bfa', alpha=0.85, edgecolor='none')
    axes[1].set_title('Distributie varsta estimata'); axes[1].set_xlabel('Varsta')
    plt.tight_layout()
    plt.show()

## 8. Vocabular

In [ ]:
vocab = pd.read_sql("""
    SELECT word, COUNT(*) AS freq
    FROM words w
    JOIN segments s USING (segment_id)
    WHERE COALESCE(s.review_status, '') != 'rejected'
    GROUP BY word
    ORDER BY freq DESC
    LIMIT 50
""", conn)

total_words = conn.execute("""
    SELECT COUNT(*) FROM words w
    JOIN segments s USING (segment_id)
    WHERE COALESCE(s.review_status, '') != 'rejected'
""").fetchone()[0]

unique_words = conn.execute("""
    SELECT COUNT(DISTINCT word) FROM words w
    JOIN segments s USING (segment_id)
    WHERE COALESCE(s.review_status, '') != 'rejected'
""").fetchone()[0]

print(f'Total cuvinte: {total_words:,}')
print(f'Vocabular unic: {unique_words:,}')
print(f'\nTop 20 cuvinte:')
print(vocab.head(20).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(vocab['word'][:20], vocab['freq'][:20], color='#34d399', alpha=0.85)
ax.set_title('Top 20 cuvinte frecvente'); ax.set_ylabel('Frecventa')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 9. Interogare liberă

In [ ]:
# Schimba query-ul cum vrei
query = """
SELECT quality_tier, review_status, COUNT(*) AS n
FROM segments
GROUP BY 1, 2
ORDER BY 1, 2
"""

pd.read_sql(query, conn)